# CN7023 – Artificial Intelligence & Machine Vision
# Malaria Cell Image Classification: A Progressive Deep Learning Study
### NIH Malaria Cell Images Dataset (Kaggle)
### Three-Model Pipeline: Custom CNN → EfficientNetB3 → Vision Transformer
---
**Running on Kaggle Notebooks**
- Go to the **Data** panel (right sidebar) → Add Data → search `cell-images-for-detecting-malaria` → Add
- Set accelerator: Settings → Accelerator → **GPU P100**
- Then: Run All

**Notebook Structure:**
1. Setup & GPU Check
2. Dataset Path (auto-detected from Kaggle)
3. Data Exploration & Visualisation
4. Preprocessing & Augmentation Pipeline
5. Model 1 – Custom CNN (Baseline)
6. Model 2 – EfficientNetB3 Transfer Learning
7. Model 3 – Vision Transformer (ViT)
8. Results: Accuracy Curves, Confusion Matrices, Test Accuracy
9. Grad-CAM Explainability Visualisation
10. Model Comparison Summary

## 1. Setup & GPU Check

In [ ]:
# Install any missing packages (most are pre-installed on Kaggle)
import subprocess
subprocess.run(['pip', 'install', '-q', 'seaborn'], capture_output=True)

import os, glob, shutil, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score
)

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU        : {gpus[0].name}')
    print(f'GPU memory : {tf.config.experimental.get_memory_info("GPU:0")["current"] / 1e9:.1f} GB used')
else:
    print('GPU NOT FOUND — go to Settings > Accelerator > GPU P100')

## 2. Dataset Path

**Kaggle setup (one-time):**
1. Click **+ Add Data** in the right sidebar
2. Search for `cell-images-for-detecting-malaria`
3. Click **Add** — the dataset mounts automatically at `/kaggle/input/`
4. Then run this cell to confirm the path is found.

In [ ]:
# Auto-detect dataset path — works on Kaggle without any API key
import os

# Kaggle datasets mount at /kaggle/input/<dataset-name>/
KAGGLE_INPUT = '/kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images'
KAGGLE_INPUT_ALT = '/kaggle/input/cell-images-for-detecting-malaria/cell_images'

if os.path.exists(KAGGLE_INPUT):
    DATA_DIR = KAGGLE_INPUT
elif os.path.exists(KAGGLE_INPUT_ALT):
    DATA_DIR = KAGGLE_INPUT_ALT
else:
    # Fallback: search for the folders
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'Parasitized' in dirs and 'Uninfected' in dirs:
            DATA_DIR = root
            break
    else:
        raise FileNotFoundError(
            'Dataset not found!\n'
            'Please add it via: + Add Data > cell-images-for-detecting-malaria'
        )

PARASITIZED_DIR = os.path.join(DATA_DIR, 'Parasitized')
UNINFECTED_DIR  = os.path.join(DATA_DIR, 'Uninfected')

print(f'Dataset found at : {DATA_DIR}')
print(f'Parasitized      : {len(os.listdir(PARASITIZED_DIR)):,} files')
print(f'Uninfected       : {len(os.listdir(UNINFECTED_DIR)):,} files')

In [ ]:
# Set up working directory for outputs
# On Kaggle, /kaggle/working/ is writable and persists during the session
WORK_DIR = '/kaggle/working'
os.makedirs(f'{WORK_DIR}/split/train/Parasitized', exist_ok=True)
os.makedirs(f'{WORK_DIR}/split/train/Uninfected',  exist_ok=True)
os.makedirs(f'{WORK_DIR}/split/val/Parasitized',   exist_ok=True)
os.makedirs(f'{WORK_DIR}/split/val/Uninfected',    exist_ok=True)
os.makedirs(f'{WORK_DIR}/split/test/Parasitized',  exist_ok=True)
os.makedirs(f'{WORK_DIR}/split/test/Uninfected',   exist_ok=True)
print(f'Working directory: {WORK_DIR}')
print('Split directories created.')

## 3. Data Exploration & Visualisation

In [ ]:
parasitized_images = sorted(glob.glob(os.path.join(PARASITIZED_DIR, '*.png')))
uninfected_images  = sorted(glob.glob(os.path.join(UNINFECTED_DIR,  '*.png')))

# Filter out non-image files (dataset contains a stray Thumbs.db)
parasitized_images = [p for p in parasitized_images if os.path.getsize(p) > 1000]
uninfected_images  = [u for u in uninfected_images  if os.path.getsize(u) > 1000]
total = len(parasitized_images) + len(uninfected_images)

print('=' * 55)
print('  DATASET SUMMARY')
print('=' * 55)
print(f'  Parasitized images  : {len(parasitized_images):,}')
print(f'  Uninfected images   : {len(uninfected_images):,}')
print(f'  Total images        : {total:,}')
print(f'  Class balance       : perfectly balanced (50/50)')
print(f'  Source              : NIH / Kaggle')
print('=' * 55)

# Check image sizes
sizes = [Image.open(p).size for p in parasitized_images[:100]]
ws, hs = zip(*sizes)
print(f'\nImage dimensions (sample of 100):')
print(f'  Width  — min:{min(ws)} max:{max(ws)} mean:{np.mean(ws):.0f}px')
print(f'  Height — min:{min(hs)} max:{max(hs)} mean:{np.mean(hs):.0f}px')
print(f'  → Variable size: resizing to 128x128 (CNN) / 224x224 (EfficientNet/ViT)')

In [ ]:
# ── Figure 1: Class Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('NIH Malaria Cell Images – Class Distribution', fontsize=14, fontweight='bold')

classes = ['Parasitized', 'Uninfected']
counts  = [len(parasitized_images), len(uninfected_images)]
colours = ['#e74c3c', '#27ae60']

axes[0].bar(classes, counts, color=colours, edgecolor='black', width=0.45)
axes[0].set_ylabel('Number of Images')
axes[0].set_title('Image Count per Class')
axes[0].set_ylim(0, 16000)
for i, v in enumerate(counts):
    axes[0].text(i, v + 150, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts, labels=classes, colors=colours, autopct='%1.1f%%',
            startangle=90, explode=(0.04, 0.04))
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.savefig('fig1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig1_class_distribution.png')

In [ ]:
# ── Figure 2: Sample Images ──
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Sample Cell Images — NIH Malaria Dataset', fontsize=14, fontweight='bold')

for i in range(6):
    img = Image.open(parasitized_images[i*10]).convert('RGB').resize((128,128))
    axes[0,i].imshow(img)
    axes[0,i].set_title('Parasitized', color='#e74c3c', fontsize=9, fontweight='bold')
    axes[0,i].axis('off')

for i in range(6):
    img = Image.open(uninfected_images[i*10]).convert('RGB').resize((128,128))
    axes[1,i].imshow(img)
    axes[1,i].set_title('Uninfected', color='#27ae60', fontsize=9, fontweight='bold')
    axes[1,i].axis('off')

plt.tight_layout()
plt.savefig('fig2_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig2_sample_images.png')

## 4. Preprocessing & Augmentation Pipeline

**Steps:**
- Resize: 128×128 (CNN) and 224×224 (EfficientNet / ViT)
- Normalise pixel values to [0, 1] (divide by 255)
- Data split: 70% train / 15% validation / 15% test (stratified)
- Augmentation on training set only: horizontal flip, vertical flip, rotation ±20°, zoom 15%, width/height shift 10%

In [ ]:
# ── Configuration ──
IMG_SIZE_CNN = (128, 128)
IMG_SIZE_TL  = (224, 224)
BATCH_SIZE   = 32

# Build labelled lists
all_paths  = parasitized_images + uninfected_images
all_labels = ['Parasitized'] * len(parasitized_images) + ['Uninfected'] * len(uninfected_images)

# Stratified 70 / 15 / 15 split
X_train, X_tmp, y_train, y_tmp = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

print(f'Train      : {len(X_train):,} images')
print(f'Validation : {len(X_val):,} images')
print(f'Test       : {len(X_test):,} images')

In [ ]:
# Organise into folder structure for ImageDataGenerator
# Copies are made into /kaggle/working/split/ which is writable
def build_split_dirs(paths, labels, split_name):
    base = f'{WORK_DIR}/split/{split_name}'
    for cls in ['Parasitized', 'Uninfected']:
        os.makedirs(f'{base}/{cls}', exist_ok=True)
    for src, lbl in zip(paths, labels):
        dst = f'{base}/{lbl}/{os.path.basename(src)}'
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

print('Building split directories (copying ~27k images)...')
print('This takes ~2 minutes on Kaggle P100 — please wait...')
build_split_dirs(X_train, y_train, 'train')
build_split_dirs(X_val,   y_val,   'val')
build_split_dirs(X_test,  y_test,  'test')
print('Done.')

In [ ]:
# ── ImageDataGenerators ──
# Training: normalise + augment
train_aug = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    fill_mode='nearest'
)
# Validation / Test: normalise only
val_gen_factory = ImageDataGenerator(rescale=1./255)

def make_gen(gen_obj, split, img_size, shuffle=True):
    return gen_obj.flow_from_directory(
        f'{WORK_DIR}/split/{split}',
        target_size=img_size,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        shuffle=shuffle,
        seed=42
    )

# CNN generators (128x128)
cnn_train_gen = make_gen(train_aug,        'train', IMG_SIZE_CNN)
cnn_val_gen   = make_gen(val_gen_factory,  'val',   IMG_SIZE_CNN, shuffle=False)
cnn_test_gen  = make_gen(val_gen_factory,  'test',  IMG_SIZE_CNN, shuffle=False)

# Transfer learning generators (224x224)
tl_train_gen  = make_gen(train_aug,        'train', IMG_SIZE_TL)
tl_val_gen    = make_gen(val_gen_factory,  'val',   IMG_SIZE_TL,  shuffle=False)
tl_test_gen   = make_gen(val_gen_factory,  'test',  IMG_SIZE_TL,  shuffle=False)

print(f'Class indices: {cnn_train_gen.class_indices}')
print('Generators ready.')

In [ ]:
# ── Figure 3: Augmented Samples ──
batch_imgs, batch_labels = next(cnn_train_gen)
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Augmented Training Images (Post-Preprocessing)', fontsize=13, fontweight='bold')
class_names = {0: 'Parasitized', 1: 'Uninfected'}
for i, ax in enumerate(axes.flatten()):
    ax.imshow(batch_imgs[i])
    lbl = class_names[int(batch_labels[i])]
    ax.set_title(lbl, fontsize=8,
                 color='#e74c3c' if lbl=='Parasitized' else '#27ae60')
    ax.axis('off')
plt.tight_layout()
plt.savefig('fig3_augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_augmented_samples.png')

## 5. Model 1 – Custom CNN (Baseline)

A custom 3-block CNN trained from scratch. Establishes a performance baseline and demonstrates what is achievable without pre-trained knowledge.

**Architecture:** Input(128×128×3) → [Conv2D→BN→MaxPool→Dropout] ×3 → Flatten → Dense(256) → Dropout → Dense(1, sigmoid)

In [ ]:
def build_custom_cnn(input_shape=(128,128,3)):
    """Custom 3-block CNN baseline."""
    model = models.Sequential(name='Custom_CNN')

    # Block 1 — learns low-level features (edges, blobs)
    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same',
                            input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Block 2 — learns mid-level features (shapes, textures)
    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Block 3 — learns high-level features (parasite-specific patterns)
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Classifier head
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.50))
    model.add(layers.Dense(1, activation='sigmoid'))

    return model

cnn_model = build_custom_cnn()
cnn_model.summary()
print(f'\nTotal params: {cnn_model.count_params():,}')

In [ ]:
cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=6,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(f'{WORK_DIR}/best_cnn.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=0)
]

print('Training Custom CNN (baseline)...')
cnn_history = cnn_model.fit(
    cnn_train_gen,
    epochs=25,
    validation_data=cnn_val_gen,
    callbacks=cnn_callbacks,
    verbose=1
)

## 6. Model 2 – EfficientNetB3 Transfer Learning

EfficientNetB3 pre-trained on ImageNet (1.2M images, 1000 classes). Uses compound scaling to balance depth, width and resolution — achieving higher accuracy with fewer parameters than ResNet50.

**Two-phase training:**
1. Phase 1 — freeze all EfficientNet layers, train new classification head only
2. Phase 2 — unfreeze top layers, fine-tune with very low learning rate (1e-5)

In [ ]:
def build_efficientnet(input_shape=(224,224,3)):
    """EfficientNetB3 with custom classification head."""
    base = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base.trainable = False  # Freeze for Phase 1

    inputs = keras.Input(shape=input_shape)
    # Note: EfficientNet includes its own preprocessing internally
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs, name='EfficientNetB3_TL')
    return model, base

eff_model, eff_base = build_efficientnet()
eff_model.summary()
print(f'\nTotal params        : {eff_model.count_params():,}')
print(f'Trainable params    : {sum(tf.size(v).numpy() for v in eff_model.trainable_variables):,}')

In [ ]:
# ── Phase 1: Train head only ──
eff_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_p1 = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1)
]

print('Phase 1: Training classification head (EfficientNetB3 frozen)...')
hist_p1 = eff_model.fit(
    tl_train_gen, epochs=10,
    validation_data=tl_val_gen,
    callbacks=callbacks_p1, verbose=1
)

In [ ]:
# ── Phase 2: Fine-tune top 30 layers ──
eff_base.trainable = True
for layer in eff_base.layers[:-30]:
    layer.trainable = False

n_trainable = sum(1 for l in eff_base.layers if l.trainable)
print(f'Fine-tuning {n_trainable} layers.')

# Lower LR for fine-tuning — prevents destroying pre-trained weights
eff_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint(f'{WORK_DIR}/best_efficientnet.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0)
]

print('Phase 2: Fine-tuning EfficientNetB3 top layers...')
hist_p2 = eff_model.fit(
    tl_train_gen, epochs=15,
    validation_data=tl_val_gen,
    callbacks=callbacks_p2, verbose=1
)

# Merge histories for plotting
eff_history = {k: hist_p1.history[k] + hist_p2.history[k]
               for k in hist_p1.history}
print(f'Total EfficientNet epochs: {len(eff_history["accuracy"])}')

## 7. Model 3 – Vision Transformer (ViT)

A lightweight ViT implemented in Keras. Splits the image into 16×16 patches, embeds each patch, adds positional encoding, then applies multi-head self-attention layers. The [CLS] token aggregates global information for the final classification decision.

This represents the frontier of computer vision — surpassing CNNs by learning global spatial relationships rather than local convolution patterns.

In [ ]:
# ── Vision Transformer Implementation ──

class PatchEmbedding(layers.Layer):
    """Splits image into patches and linearly embeds each patch."""
    def __init__(self, patch_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim  = embed_dim
        self.projection = layers.Dense(embed_dim)

    def call(self, x):
        B, H, W, C = tf.shape(x)[0], x.shape[1], x.shape[2], x.shape[3]
        # Reshape into patches: (B, num_patches, patch_h*patch_w*C)
        p = self.patch_size
        x = tf.reshape(x, [B, H//p, p, W//p, p, C])
        x = tf.transpose(x, [0,1,3,2,4,5])
        x = tf.reshape(x, [B, (H//p)*(W//p), p*p*C])
        return self.projection(x)


class TransformerBlock(layers.Layer):
    """Single transformer encoder block: multi-head attention + MLP."""
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn  = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim//num_heads, dropout=dropout)
        self.mlp   = models.Sequential([
            layers.Dense(mlp_dim, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout)
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=False):
        # Self-attention with residual connection
        attn_out = self.attn(x, x, training=training)
        x = self.norm1(x + attn_out)
        # MLP with residual connection
        mlp_out = self.mlp(x, training=training)
        return self.norm2(x + mlp_out)


def build_vit(
    img_size=224, patch_size=16, embed_dim=256,
    num_heads=8, num_layers=6, mlp_dim=512, dropout=0.1
):
    """Lightweight Vision Transformer for binary classification."""
    num_patches  = (img_size // patch_size) ** 2  # 196 for 224px/16px patches

    inputs = keras.Input(shape=(img_size, img_size, 3))

    # 1. Patch embedding
    x = PatchEmbedding(patch_size, embed_dim)(inputs)

    # 2. Prepend learnable [CLS] token
    cls_token = tf.Variable(
        tf.zeros([1, 1, embed_dim]), trainable=True, name='cls_token')
    cls_tokens = tf.broadcast_to(cls_token, [tf.shape(x)[0], 1, embed_dim])
    x = tf.concat([cls_tokens, x], axis=1)  # (B, num_patches+1, embed_dim)

    # 3. Add learnable positional embedding
    pos_embed = tf.Variable(
        tf.random.normal([1, num_patches+1, embed_dim], stddev=0.02),
        trainable=True, name='pos_embed')
    x = x + pos_embed
    x = layers.Dropout(dropout)(x)

    # 4. Stack transformer encoder blocks
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim, dropout)(x)

    # 5. Extract [CLS] token and classify
    cls_out = x[:, 0, :]                     # (B, embed_dim)
    cls_out = layers.LayerNormalization(epsilon=1e-6)(cls_out)
    cls_out = layers.Dense(128, activation='gelu')(cls_out)
    cls_out = layers.Dropout(0.3)(cls_out)
    outputs = layers.Dense(1, activation='sigmoid')(cls_out)

    return keras.Model(inputs, outputs, name='VisionTransformer')


vit_model = build_vit()
vit_model.summary()
print(f'\nTotal ViT params: {vit_model.count_params():,}')

In [ ]:
# Cosine decay learning rate schedule (standard for ViT training)
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps     = steps_per_epoch * 20
warmup_steps    = steps_per_epoch * 3

lr_schedule = CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=total_steps - warmup_steps,
    alpha=1e-6
)

vit_model.compile(
    optimizer=Adam(learning_rate=lr_schedule),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

vit_callbacks = [
    EarlyStopping(monitor='val_loss', patience=7,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(f'{WORK_DIR}/best_vit.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0)
]

print('Training Vision Transformer...')
vit_history = vit_model.fit(
    tl_train_gen, epochs=20,
    validation_data=tl_val_gen,
    callbacks=vit_callbacks,
    verbose=1
)

## 8. Results

In [ ]:
# ── Helper: Plot training curves ──
def plot_curves(history_dict, title, save_path):
    epochs = range(1, len(history_dict['accuracy']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    # Accuracy
    ax1.plot(epochs, [v*100 for v in history_dict['accuracy']],
             'b-o', label='Training', markersize=3)
    ax1.plot(epochs, [v*100 for v in history_dict['val_accuracy']],
             'r-s', label='Validation', markersize=3)
    ax1.set_title('Accuracy Curve')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy (%)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(epochs, history_dict['loss'],     'b-o', label='Training', markersize=3)
    ax2.plot(epochs, history_dict['val_loss'], 'r-s', label='Validation', markersize=3)
    ax2.set_title('Loss Curve')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

plot_curves(cnn_history.history,   'Custom CNN (Baseline) — Training History',          'fig4_cnn_curves.png')
plot_curves(eff_history,           'EfficientNetB3 Transfer Learning — Training History','fig5_eff_curves.png')
plot_curves(vit_history.history,   'Vision Transformer — Training History',              'fig6_vit_curves.png')

In [ ]:
# ── Test Set Evaluation ──
print('Evaluating all models on held-out test set...\n')

def evaluate_model(model, test_gen, name):
    test_gen.reset()
    loss, acc = model.evaluate(test_gen, verbose=0)
    test_gen.reset()
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = (y_prob > 0.5).astype(int).flatten()
    y_true = test_gen.classes
    return {
        'name':      name,
        'accuracy':  accuracy_score(y_true, y_pred) * 100,
        'precision': precision_score(y_true, y_pred) * 100,
        'recall':    recall_score(y_true, y_pred) * 100,
        'f1':        f1_score(y_true, y_pred) * 100,
        'loss':      loss,
        'y_pred':    y_pred,
        'y_true':    y_true
    }

cnn_res = evaluate_model(cnn_model,  cnn_test_gen, 'Custom CNN')
eff_res = evaluate_model(eff_model,  tl_test_gen,  'EfficientNetB3 TL')
vit_res = evaluate_model(vit_model,  tl_test_gen,  'Vision Transformer')

print('=' * 62)
print(f'  TEST SET RESULTS')
print('=' * 62)
print(f"  {'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>7}")
print('-' * 62)
for r in [cnn_res, eff_res, vit_res]:
    print(f"  {r['name']:<22} {r['accuracy']:>8.2f}% {r['precision']:>9.2f}% "
          f"{r['recall']:>7.2f}% {r['f1']:>6.2f}%")
print('=' * 62)

In [ ]:
# ── Confusion Matrices ──
def plot_cm(result, save_path):
    class_names = ['Parasitized', 'Uninfected']
    cm = confusion_matrix(result['y_true'], result['y_pred'])
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_title(
        f"{result['name']}\nConfusion Matrix — Test Accuracy: {result['accuracy']:.2f}%",
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')
    print(classification_report(
        result['y_true'], result['y_pred'], target_names=class_names))

plot_cm(cnn_res, 'fig7_cnn_cm.png')
plot_cm(eff_res, 'fig8_eff_cm.png')
plot_cm(vit_res, 'fig9_vit_cm.png')

## 9. Grad-CAM Explainability Visualisation

Grad-CAM (Gradient-weighted Class Activation Mapping) generates a heatmap showing **which regions of the cell image** the model focuses on when making its classification decision. Warm colours (red/yellow) indicate high attention; cool colours (blue) indicate low attention.

This provides clinical insight: the model should attend to the purple stained parasite regions in infected cells.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """
    Computes Grad-CAM heatmap for a single image.
    img_array: (1, H, W, 3) normalised image
    """
    # Build a model that outputs (last conv activations, final prediction)
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output,
                 model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        # For binary classification, use the output directly
        loss = predictions[:, 0]

    # Gradient of prediction w.r.t. last conv layer
    grads = tape.gradient(loss, conv_outputs)

    # Pool gradients over spatial dimensions
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight conv output channels by pooled gradients
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # ReLU + normalise
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_path, heatmap, alpha=0.45):
    """Overlays Grad-CAM heatmap on original image."""
    img = np.array(Image.open(img_path).convert('RGB').resize((224,224)))
    heatmap_resized = np.uint8(255 * heatmap)
    heatmap_colour  = cm.jet(heatmap_resized)[:, :, :3]
    heatmap_colour  = np.uint8(heatmap_colour * 255)
    # Resize heatmap to image size
    from PIL import Image as PILImage
    hm_pil = PILImage.fromarray(heatmap_colour).resize((224,224))
    hm_arr = np.array(hm_pil)
    superimposed = np.uint8(img * (1 - alpha) + hm_arr * alpha)
    return img, superimposed

print('Grad-CAM functions defined.')

# Find the last conv layer name in the CNN model
last_conv = [l.name for l in cnn_model.layers if 'conv' in l.name][-1]
print(f'Last conv layer in Custom CNN: {last_conv}')

In [ ]:
# ── Figure 10: Grad-CAM on Custom CNN ──
# Select representative samples
sample_parasitized = parasitized_images[5]
sample_uninfected  = uninfected_images[5]

def get_gradcam_pair(img_path, model, conv_layer):
    img = np.array(Image.open(img_path).convert('RGB').resize((128,128))) / 255.
    img_arr = np.expand_dims(img, 0).astype(np.float32)
    heatmap = make_gradcam_heatmap(img_arr, model, conv_layer)
    # Resize heatmap to 128x128 for CNN
    from PIL import Image as PILImage
    hm_pil  = PILImage.fromarray(np.uint8(255*heatmap)).resize((128,128))
    hm_arr  = np.array(hm_pil) / 255.
    hm_col  = cm.jet(hm_arr)[:,:,:3]
    overlay = np.uint8((np.array(Image.open(img_path).convert('RGB').resize((128,128)))
                        * 0.55 + hm_col * 255 * 0.45).clip(0,255))
    return np.array(Image.open(img_path).convert('RGB').resize((128,128))), overlay

orig_p, cam_p = get_gradcam_pair(sample_parasitized, cnn_model, last_conv)
orig_u, cam_u = get_gradcam_pair(sample_uninfected,  cnn_model, last_conv)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
fig.suptitle('Grad-CAM Explainability — Custom CNN\nWarm colours = where the model looks',
             fontsize=13, fontweight='bold')

axes[0,0].imshow(orig_p); axes[0,0].set_title('Parasitized — Original', color='#e74c3c'); axes[0,0].axis('off')
axes[0,1].imshow(cam_p);  axes[0,1].set_title('Parasitized — Grad-CAM', color='#e74c3c'); axes[0,1].axis('off')
axes[1,0].imshow(orig_u); axes[1,0].set_title('Uninfected — Original',  color='#27ae60'); axes[1,0].axis('off')
axes[1,1].imshow(cam_u);  axes[1,1].set_title('Uninfected — Grad-CAM',  color='#27ae60'); axes[1,1].axis('off')

plt.tight_layout()
plt.savefig('fig10_gradcam_cnn.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig10_gradcam_cnn.png')

In [ ]:
# ── Grad-CAM on EfficientNetB3 — find its last conv layer ──
eff_conv_layer = [l.name for l in eff_model.layers
                  if isinstance(l, keras.layers.Conv2D)][-1]
# EfficientNet wraps layers — find within the base model
eff_base_last_conv = [l.name for l in eff_base.layers
                      if isinstance(l, keras.layers.Conv2D)][-1]
print(f'EfficientNet last conv: {eff_base_last_conv}')

# Build a flat model for Grad-CAM access
eff_flat = keras.Sequential([
    keras.Input(shape=(224,224,3)),
    eff_model
], name='eff_flat')

# Find suitable layer — use top_conv which is the last conv in EfficientNet
try:
    # EfficientNetB3 last meaningful conv is 'top_conv'
    _ = eff_model.get_layer('top_conv')
    eff_cam_layer = 'top_conv'
except:
    eff_cam_layer = eff_base_last_conv
print(f'Using layer for EfficientNet Grad-CAM: {eff_cam_layer}')

## 10. Final Comparison Summary

In [ ]:
# ── Figure 11: Side-by-side metric comparison ──
results = [cnn_res, eff_res, vit_res]
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metric_keys  = ['accuracy', 'precision', 'recall', 'f1']
colours_bar  = ['#3498db', '#27ae60', '#f39c12']

x = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (res, col) in enumerate(zip(results, colours_bar)):
    vals  = [res[k] for k in metric_keys]
    bars  = ax.bar(x + (i - 1) * width, vals, width,
                   label=res['name'], color=col, edgecolor='black', linewidth=0.5)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.2,
                f'{bar.get_height():.1f}%',
                ha='center', va='bottom', fontsize=8)

ax.set_title('Model Performance Comparison — Test Set', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylabel('Score (%)')
ax.set_ylim(80, 102)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig11_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig11_comparison.png')

In [ ]:
# ── Print complete summary table ──
print('\n' + '=' * 68)
print('  FINAL MODEL COMPARISON SUMMARY')
print('=' * 68)
print(f"  {'Model':<22} {'Acc':>7} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Params':>12}")
print('-' * 68)

param_counts = {
    'Custom CNN':        cnn_model.count_params(),
    'EfficientNetB3 TL': eff_model.count_params(),
    'Vision Transformer':vit_model.count_params()
}

for r in results:
    p = param_counts[r['name']]
    print(f"  {r['name']:<22} {r['accuracy']:>6.2f}% {r['precision']:>7.2f}% "
          f"{r['recall']:>7.2f}% {r['f1']:>7.2f}% {p:>12,}")

print('=' * 68)
best = max(results, key=lambda r: r['accuracy'])
print(f"\n  Best model: {best['name']} with {best['accuracy']:.2f}% test accuracy")

In [ ]:
# ── List all saved figures ──
all_figs = [
    'fig1_class_distribution.png',
    'fig2_sample_images.png',
    'fig3_augmented_samples.png',
    'fig4_cnn_curves.png',
    'fig5_eff_curves.png',
    'fig6_vit_curves.png',
    'fig7_cnn_cm.png',
    'fig8_eff_cm.png',
    'fig9_vit_cm.png',
    'fig10_gradcam_cnn.png',
    'fig11_comparison.png'
]
print('Generated figures:')
for f in all_figs:
    status = 'saved' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}] {f}')

print('\nTo download all figures:')
print("  from google.colab import files")
print("  for f in all_figs: files.download(f)")

In [ ]:
# ── Output files ──
# On Kaggle, all files saved to /kaggle/working/ are automatically
# available for download from the Output tab on the right sidebar.
# No manual download step needed!

print('All figures and models are saved to /kaggle/working/')
print('To download: click the Output tab in the right sidebar.')
print()

import os
output_files = [f for f in os.listdir(WORK_DIR)
                if f.endswith(('.png', '.keras', '.h5'))]
print(f'Files available for download ({len(output_files)} total):')
for f in sorted(output_files):
    size_kb = os.path.getsize(f'{WORK_DIR}/{f}') / 1024
    print(f'  {f:<40} {size_kb:>8.1f} KB')